In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import torch
import torch.nn as nn

from src.config import SEED
from src.transfer import build_feature_extractor, build_fine_tuner, train_transfer

torch.manual_seed(SEED)

# Transfer Learning con ResNet18
## Fase 4 — Feature Extraction y Fine-tuning

En lugar de entrenar una red desde cero como la CNN, aprovechamos **ResNet18 preentrenada en ImageNet**: una red que ya aprendió a reconocer bordes, texturas y estructuras visuales complejas en 1.2 millones de imágenes.

Usamos ImageNet específicamente porque normalizamos nuestras imágenes con sus stats (mean y std), lo que garantiza que la red recibe los datos en el rango que espera.

Comparamos dos estrategias:

| Estrategia | Qué se entrena | LR | Parámetros entrenables |
|---|---|---|---|
| **Feature extraction** | Solo clasificador (fc) | 1e-3 | ~513 |
| **Fine-tuning** | layer4 + clasificador | 1e-4 | ~2.1M |

**¿Por qué fine-tuning debería ganar?**  
Detectar imágenes IA requiere reconocer artefactos de difusión y texturas sintéticas que no existen en ImageNet. El feature extraction usa los features genéricos tal como vienen; el fine-tuning adapta las capas más profundas a nuestra tarea específica.

Pipeline:
1. Feature extraction — backbone congelado, solo fc entrenable
2. Fine-tuning — layer4 + fc descongelados con LR chico

## 1. Feature Extraction

Congelamos todo el backbone de ResNet50 y reemplazamos el clasificador final (originalmente 2048 → 1000 clases ImageNet) por uno binario (2048 → 1 logit).

ResNet50 actúa como un extractor de features fijo: cada imagen se convierte en un vector de 2048 dimensiones, y el clasificador aprende a separar real de fake en ese espacio.

In [2]:
model_fe = build_feature_extractor()

total   = sum(p.numel() for p in model_fe.parameters())
trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
print(f"Parámetros totales:      {total:,}")
print(f"Parámetros entrenables:  {trainable:,}  ({100*trainable/total:.1f}%)")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/lautarocaminoa/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:14<00:00, 3.19MB/s]


Parámetros totales:      11,177,025
Parámetros entrenables:  513  (0.0%)


In [3]:
historial_fe = train_transfer(model_fe, run_name="fe")

Dispositivo: mps
Epoch   1/50  train_loss=0.4918  val_loss=0.4415  val_acc=0.7920


KeyboardInterrupt: 

## 2. Fine-tuning

Partimos del mismo ResNet18 preentrenado pero descongelamos **layer4** (el último bloque residual del backbone) además del clasificador. El resto del backbone sigue congelado.

Usamos un **learning rate más chico (1e-4)** para actualizar los pesos preentrenados con pasos pequeños. Si usáramos el mismo LR que en feature extraction, destruiríamos los features que ya aprendió ResNet18.

**¿Por qué solo layer4?**  
- layer1-3 detectan features genéricos (bordes, gradientes, texturas simples) → transfieren bien, no hace falta adaptarlos  
- layer4 detecta estructuras de alto nivel → necesita adaptarse a los patrones de imágenes IA

In [ ]:
model_ft = build_fine_tuner()

total     = sum(p.numel() for p in model_ft.parameters())
trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Parámetros totales:      {total:,}")
print(f"Parámetros entrenables:  {trainable:,}  ({100*trainable/total:.1f}%)")

print("\nCapas descongeladas:")
for name, param in model_ft.named_parameters():
    if param.requires_grad:
        print(f"  {name}")

In [ ]:
historial_ft = train_transfer(model_ft, run_name="ft", lr=1e-4)